In [1]:
import os
import h5py
from joblib import Parallel, delayed
from pathlib import Path
from typing import List

from mcts_agent import MCTS_Agent
from random_network import RandomNetwork
from self_play import generate_self_play_games
from train_epochs import train_network_epochs
from model_manager import ModelManager
from rl_agent import RLAgent
import torch

In [2]:
RANDOM_SAVE_DIR = ".\\test1\\random\\"
RANDOM_MODEL_DIR= ".\\test1\\random_model\\"


def create_dataset_file(n_thread: int) -> str:
    file_name = f"dataset_{n_thread}.h5"
    full_path = os.path.join(RANDOM_SAVE_DIR, file_name)
    
    # Добавляем параметр libver='latest' для поддержки формата 1.10+
    with h5py.File(full_path, 'w', libver='latest') as f:
        print(f"HDF5 файл успешно создан по пути: {full_path}")
        
    return full_path


def generate_random_games(path, idx):
    network = RandomNetwork(random_seed=idx)
    agent_rl = MCTS_Agent(temperature=1.0)
    agent_rl.set_random_seed(idx)
    agent_rl.set_network(network)

    results = generate_self_play_games(
        games_to_generate=400,
        rl_agent=agent_rl,
        file_path=path,
        log = False,
        write=True
    )
    return results


# Создаем функцию-обертку для воркера
def worker(i):
    path = create_dataset_file(i)
    winners = generate_random_games(path, i)
    return {'path': path, 'winners': winners}

def create_random_dataset(jobs):
    print("Начинаем генерацию...")
    results = Parallel(n_jobs=jobs, verbose=10)(delayed(worker)(i) for i in range(jobs))
    return results
    print(f"Генерация завершена! Созданы файлы: {results}")

In [3]:
jobs = 28
results = create_random_dataset(jobs)

Начинаем генерацию...


[Parallel(n_jobs=28)]: Using backend LokyBackend with 28 concurrent workers.
[Parallel(n_jobs=28)]: Done   3 out of  28 | elapsed: 58.4min remaining: 486.7min
[Parallel(n_jobs=28)]: Done   6 out of  28 | elapsed: 58.6min remaining: 215.0min
[Parallel(n_jobs=28)]: Done   9 out of  28 | elapsed: 58.8min remaining: 124.1min
[Parallel(n_jobs=28)]: Done  12 out of  28 | elapsed: 58.9min remaining: 78.5min
[Parallel(n_jobs=28)]: Done  15 out of  28 | elapsed: 59.0min remaining: 51.1min
[Parallel(n_jobs=28)]: Done  18 out of  28 | elapsed: 59.0min remaining: 32.8min
[Parallel(n_jobs=28)]: Done  21 out of  28 | elapsed: 59.1min remaining: 19.7min
[Parallel(n_jobs=28)]: Done  24 out of  28 | elapsed: 59.4min remaining:  9.9min
[Parallel(n_jobs=28)]: Done  28 out of  28 | elapsed: 59.6min finished


In [4]:
a = [winner for i in results for winner in i['winners']]

In [5]:
a = [winner for i in results for winner in i['winners']]

In [6]:
len([i for i in a if i == 1]), len([i for i in a if i == 2]), len([i for i in a if i == 0])

(5718, 5230, 252)

In [7]:
paths = [i['path'] for i in results]

In [8]:
def get_relative_file_paths(directory_path: str) -> List[str]:
    """
    Собирает все файлы в указанной директории и её поддиректориях,
    возвращая список путей в формате './папка/файл.ext'.
    """
    base_dir = Path(directory_path)
    
    # Проверяем, существует ли указанная директория
    if not base_dir.exists() or not base_dir.is_dir():
        raise ValueError(f"Директория '{directory_path}' не найдена или не является папкой.")
        
    result_paths = []
    
    # rglob('*') рекурсивно находит все файлы и папки
    for path in base_dir.rglob('*'):
        # Нас интересуют только файлы, пропускаем папки
        if path.is_file():
            # Получаем путь относительно базовой директории
            relative_path = path.relative_to(base_dir)
            
            # Формируем строку с префиксом './' и правильными прямыми слешами (posix)
            # as_posix() гарантирует, что даже на Windows будут использоваться '/', а не '\'
            formatted_path = f"{directory_path}{relative_path.as_posix()}"
            result_paths.append(formatted_path)
            
    return result_paths

In [9]:
def merge_hdf5_datasets(source_files: list, output_filename: str) -> str:
    print(f"Начинаем слияние {len(source_files)} файлов в '{output_filename}'...")

    dataset_specs = {
        'state_tensor': dict(dtype='float32', compression='lzf'),
        'mcts_policy': dict(dtype='float32', compression='lzf'),
        'territories': dict(dtype='int8', compression='lzf'),
        'value': dict(dtype='float32', compression=None),
        'score': dict(dtype='int64', compression=None),
        'turn': dict(dtype='int32', compression=None),
        'move_meta': dict(dtype='S32', compression=None),
        'game_id': dict(dtype='S32', compression=None),
        'noise_seed': dict(dtype='int32', compression=None)
    }

    total_samples_copied = 0

    with h5py.File(output_filename, 'w', libver=('v110', 'latest')) as f_out:
        initialized = False

        for file_idx, file_path in enumerate(source_files):
            if not os.path.exists(file_path):
                print(f"Файл {file_path} не найден, пропускаем.")
                continue

            print(f"Обработка файла: {file_path}...")

            with h5py.File(file_path, 'r', libver='latest') as f_in:
                missing = [name for name in dataset_specs if name not in f_in]
                if missing:
                    print(f"  Файл пропущен: отсутствуют datasets: {missing}")
                    continue

                input_len = f_in['turn'].shape[0]
                if input_len == 0:
                    print(f"  Файл пустой, пропускаем.")
                    continue

                for name in dataset_specs:
                    if f_in[name].shape[0] != input_len:
                        raise ValueError(
                            f"В файле {file_path} dataset '{name}' имеет длину {f_in[name].shape[0]}, "
                            f"ожидалась {input_len}"
                        )

                if not initialized:
                    for name, spec in dataset_specs.items():
                        src = f_in[name]
                        shape = (0,) + src.shape[1:]
                        maxshape = (None,) + src.shape[1:]

                        if name == 'state_tensor':
                            chunks = (4,) + src.shape[1:]
                        elif name == 'mcts_policy':
                            chunks = (32,) + src.shape[1:]
                        elif name == 'territories':
                            chunks = (64,) + src.shape[1:]
                        else:
                            chunks = (1024,) if len(src.shape) == 1 else (1024,) + src.shape[1:]

                        create_kwargs = {
                            'shape': shape,
                            'maxshape': maxshape,
                            'chunks': chunks,
                            'dtype': spec['dtype'],
                        }
                        if spec['compression'] is not None:
                            create_kwargs['compression'] = spec['compression']

                        f_out.create_dataset(name, **create_kwargs)

                    initialized = True

                current_size = f_out['turn'].shape[0]
                new_size = current_size + input_len

                for name in dataset_specs:
                    dset_out = f_out[name]
                    dset_out.resize((new_size,) + dset_out.shape[1:])

                for name in dataset_specs:
                    f_out[name][current_size:new_size] = f_in[name][:]

                total_samples_copied += input_len
                print(f"  -> Добавлено сэмплов: {input_len}")

        if initialized:
            for name in dataset_specs:
                f_out[name].flush()

    print(f"Слияние успешно завершено. Всего собрано сэмплов: {total_samples_copied}")
    return output_filename

In [10]:
merged_filpath ='.\\test1\\random_merged\\dataset.h5'
merge_hdf5_datasets(paths, merged_filpath)

Начинаем слияние 28 файлов в '.\test1\random_merged\dataset.h5'...
Обработка файла: .\test1\random\dataset_0.h5...
  -> Добавлено сэмплов: 14381
Обработка файла: .\test1\random\dataset_1.h5...
  -> Добавлено сэмплов: 14521
Обработка файла: .\test1\random\dataset_2.h5...
  -> Добавлено сэмплов: 14318
Обработка файла: .\test1\random\dataset_3.h5...
  -> Добавлено сэмплов: 14385
Обработка файла: .\test1\random\dataset_4.h5...
  -> Добавлено сэмплов: 14268
Обработка файла: .\test1\random\dataset_5.h5...
  -> Добавлено сэмплов: 14265
Обработка файла: .\test1\random\dataset_6.h5...
  -> Добавлено сэмплов: 14614
Обработка файла: .\test1\random\dataset_7.h5...
  -> Добавлено сэмплов: 14310
Обработка файла: .\test1\random\dataset_8.h5...
  -> Добавлено сэмплов: 14500
Обработка файла: .\test1\random\dataset_9.h5...
  -> Добавлено сэмплов: 14484
Обработка файла: .\test1\random\dataset_10.h5...
  -> Добавлено сэмплов: 14435
Обработка файла: .\test1\random\dataset_11.h5...
  -> Добавлено сэмплов: 1

'.\\test1\\random_merged\\dataset.h5'

In [11]:
LEARNING_RATE =1e-4
def train_model():
    model_manager = ModelManager(
        RLAgent, 
        save_dir=RANDOM_MODEL_DIR, 
        device="cuda")
    
    version = 0
    current_model_name = f"agent_v{version}.pth"
    initial_model = model_manager.create_new_model()
    model_manager.save_checkpoint(initial_model, None, {'version': version}, current_model_name)
    
    model = model_manager.model_class().to(model_manager.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    model_manager.load_checkpoint(current_model_name, model, optimizer)
    
    # Обучаем! (Ваш метод train_network_steps)
    train_network_epochs(model, optimizer, datasets_filepath=merged_filpath,  epochs=1, batch_size=256, device="cuda", buffer_length=500000)
    
    version += 1
    current_model_name = f"agent_v{version}.pth"
    model_manager.save_checkpoint(model, optimizer, {'version': version}, current_model_name)
    



In [12]:
train_model()

Creating a new model with random weights...
Checkpoint saved to .\test1\random_model\agent_v0.pth
Checkpoint loaded from .\test1\random_model\agent_v0.pth
Начинаем обучение на 404614 примерах из HDF5...
Эпоха 1/1: Pol = 1.8100, Val = 0.6591, Terr = 0.9791, Score = 4.1741
Checkpoint saved to .\test1\random_model\agent_v1.pth


In [12]:
def clean_games_groups(input_file: str, output_file: str = None):
    if output_file is None:
        # Создаем временный файл для перепаковки
        output_file = input_file.replace('.h5', '_cleaned.h5')
        
    print(f"Начинаем очистку файла: {input_file}")
    
    deleted_count = 0
    kept_count = 0
    
    # Открываем исходный файл на чтение и новый на запись
    with h5py.File(input_file, 'r') as f_in, h5py.File(output_file, 'w') as f_out:
        
        # Перебираем все ключи в корне файла
        for key in f_in.keys():
            # Если ключ начинается на 'games', просто пропускаем его (не копируем)
            if key.startswith('games'):
                print(f"  Удаляем группу: {key}")
                deleted_count += 1
            else:
                # Все остальные ключи (ваши UUID партий) копируем в новый файл
                f_in.copy(f_in[key], f_out, name=key)
                kept_count += 1
                
    print(f"Очистка завершена!")
    print(f"Удалено групп 'games...': {deleted_count}")
    print(f"Сохранено валидных партий: {kept_count}")
    print(f"Очищенный файл сохранен как: {output_file}")
    
    # Опционально: можно заменить старый файл новым
    os.replace(output_file, input_file)
    # print("Старый файл заменен на очищенный.")


In [13]:
path = '.\\shared_data\\dataset.h5'

In [22]:

def upgrade_dataset_to_swmr(path: str) -> None:
    temp_path = f"{path}.tmp"
    
    def get_valid_chunks(shape, maxshape, chunks):
        """Корректирует размер чанка под строгие требования HDF5 >= 1.10"""
        if chunks is None:
            return None
        
        valid_chunks = []
        for i in range(len(shape)):
            max_dim = maxshape[i] if maxshape is not None else shape[i]
            
            if max_dim is None:
                # Измерение не ограничено (None), чанк любого размера допустим
                valid_chunks.append(chunks[i])
            elif max_dim == 0:
                # Если максимальный размер измерения 0 (датасет пуст и не может расти), 
                # чанкование невозможно в принципе. Возвращаем None (отключаем чанки).
                return None
            else:
                # Чанк не может быть больше максимального размера датасета
                valid_chunks.append(min(chunks[i], max_dim))
                
        return tuple(valid_chunks)

    def copy_obj(name, obj, f_new):
        if isinstance(obj, h5py.Group):
            if name not in f_new:
                f_new.create_group(name)
            for k, v in obj.attrs.items():
                f_new[name].attrs[k] = v
                
        elif isinstance(obj, h5py.Dataset):
            # Исправляем чанки с учетом жестких лимитов новой версии HDF5
            valid_chunks = get_valid_chunks(obj.shape, obj.maxshape, obj.chunks)
            
            kwargs = {
                'shape': obj.shape,
                'dtype': obj.dtype,
                'chunks': valid_chunks,
                'compression': obj.compression,
                'compression_opts': obj.compression_opts,
                'shuffle': obj.shuffle,
                'fletcher32': obj.fletcher32,
                'maxshape': obj.maxshape,
                'fillvalue': obj.fillvalue,
                'scaleoffset': getattr(obj, 'scaleoffset', None)
            }
            # Убираем параметры, которых не было у оригинала
            kwargs = {k: v for k, v in kwargs.items() if v is not None}
            
            # Создаем датасет в формате v110 (Superblock >= 3)
            ds = f_new.create_dataset(name, **kwargs)
            
            # Переносим данные
            if obj.shape and all(s > 0 for s in obj.shape):
                if obj.chunks:
                    # Для чанкованных датасетов копируем по чанкам
                    for chunk_slice in obj.iter_chunks():
                        ds[chunk_slice] = obj[chunk_slice]
                else:
                    # Для нечанкованных копируем батчами, чтобы не переполнить RAM
                    if len(obj.shape) > 0:
                        row_size_bytes = (obj.size // obj.shape[0]) * obj.dtype.itemsize
                        batch_size = max(1, (100 * 1024 * 1024) // max(1, row_size_bytes))
                        for i in range(0, obj.shape[0], batch_size):
                            ds[i:i+batch_size] = obj[i:i+batch_size]
                    else:
                        ds[...] = obj[...] # Для скаляров
                        
            # Переносим атрибуты датасета
            for k, v in obj.attrs.items():
                ds.attrs[k] = v

    try:
        with h5py.File(path, 'r') as f_old:
            with h5py.File(temp_path, 'w', libver=('v110', 'latest')) as f_new:
                
                # Атрибуты корня
                for k, v in f_old.attrs.items():
                    f_new.attrs[k] = v
                
                # Обход и копирование всех элементов
                f_old.visititems(lambda name, obj: copy_obj(name, obj, f_new))
                
                # Включаем SWMR перед закрытием
                f_new.swmr_mode = True
                
        os.replace(temp_path, path)
        print(f"Файл {path} успешно пересоздан с поддержкой SWMR!")
        
    except Exception as e:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        raise RuntimeError(f"Ошибка при пересоздании датасета: {e}")

In [23]:
upgrade_dataset_to_swmr(path)

Файл .\shared_data\dataset.h5 успешно пересоздан с поддержкой SWMR!
